In [11]:
!py -m pip install langgraph-checkpoint-sqlite

# !py -m pip install --upgrade --quiet  langchain langchain-community langchainhub langchain-openai langchain-chroma bs4

  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 2.0.9
    Uninstalling langgraph-checkpoint-2.0.9:
      Successfully uninstalled langgraph-checkpoint-2.0.9



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
import pandas as pd
import os
import shutil
import numpy as np
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader, UnstructuredWordDocumentLoader
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.docstore.document import Document
from typing import List, Tuple
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langgraph.store.base import BaseStore
from langchain_core.messages import SystemMessage, HumanMessage
import re
from typing import Annotated, TypedDict
import operator
from langchain_core.documents import Document

In [2]:
OPENAI_DEPLOYMENT_ENDPOINT = "https://advancedanalyticsopenaikey.openai.azure.com/"
OPENAI_DEPLOYMENT_ENDPOINT_embed = "https://pkl-aa-dev-aiservices.openai.azure.com/" 
OPENAI_API_KEY = "FqFd4DBx1W97MSVjcZvdQsmQlhI80hXjl48iWYmZ4W3NutUlWvf0JQQJ99BDACYeBjFXJ3w3AAABACOGl3xo" 
OPENAI_API_VERSION = "2024-12-01-preview"
OPENAI_API_KEY_EMBEDDINGS = "AXEC3y1jC9ZNGCBB12NZwrpBSzScq1esexgvCXiqw7PaHE04vSMbJQQJ99BDACYeBjFXJ3w3AAABACOG4CMN" 
OPENAI_DEPLOYMENT_NAME = "gpt-4o"
OPENAI_MODEL_NAME="gpt-4o"
embedding_api_version = "2024-02-01"

# OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
# CHROMA_PATH = "c:\\Users\\SujaySunilNagvekar\\VM\\GEN AI\\SERFF\\vector_db\\testing_db"
DB_PATH  = "C:\\Users\\SujaySunilNagvekar\\VM\\GEN AI\\KM\\vector_db"
DATA_FOLDER = "C:\\Users\\SujaySunilNagvekar\\VM\\GEN AI\\KM\\Documents"

### Cohere API key 
# cohere_API_key = "wHyiTViP32Y3Q8Qwhjmd4QGNCkYNpxqtsemtSri3"
# co = cohere.Client(cohere_API_key)

In [3]:

llm = AzureChatOpenAI(
                        temperature=0.7,
                        deployment_name=OPENAI_DEPLOYMENT_NAME,
                        model_name=OPENAI_MODEL_NAME,
                        azure_endpoint=OPENAI_DEPLOYMENT_ENDPOINT,
                        openai_api_version=OPENAI_API_VERSION,
                        openai_api_key=OPENAI_API_KEY            
                    )

embeddings = AzureOpenAIEmbeddings(
                        deployment="text-embedding-3-small",
                        model="text-embedding-3-small",
                        azure_endpoint=OPENAI_DEPLOYMENT_ENDPOINT_embed,
                        openai_api_version=embedding_api_version,
                        openai_api_key=OPENAI_API_KEY_EMBEDDINGS)


In [5]:
import pandas as pd
import numpy as np

# === 1) Load your file ===
path = r"C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv"
df = pd.read_csv(path)

# (Optional) set a seed for reproducibility
# np.random.seed(42)

n = len(df)

# === 2) fault_rating: integers 1–5 uniformly at random ===
df["fault_rating"] = np.random.randint(1, 6, size=n)

# === 3) witness_available: "Yes"/"No" at random ===
df["witness_available"] = np.random.choice(["Yes", "No"], size=n)

# === 4) Time_to_Report: integers 0–7 (days) at random ===
df["Time_to_Report"] = np.random.randint(0, 8, size=n)

# === 5) subbrogation_score (spelled as requested) ===
# Based on subro_opportunity: if "Yes" → random in {0.1, 0.2, ..., 1.0}; if "No"/missing → 0.
yes_like = {"yes", "y", "true", "1"}
def to_bool_yes(v):
    if pd.isna(v):
        return False
    return str(v).strip().lower() in yes_like

mask_yes = df["Subro Opportunity"].apply(to_bool_yes) if "Subro Opportunity" in df.columns else pd.Series([False]*n)
# random integers 1..10 then divide by 10 → 0.1 .. 1.0
rand_scores = np.random.randint(1, 11, size=n) / 10.0
df["subrogation_score"] = np.where(mask_yes, rand_scores, 0.0)

# === 6) Save (updates the same file) ===
df.to_csv(path, index=False)
print("Columns added: fault_rating, witness_available, Time_to_Report, subrogation_score")
print(f"Saved to: {path}")


Columns added: fault_rating, witness_available, Time_to_Report, subrogation_score
Saved to: C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv


In [26]:
import os
import time
CSV_PATH = r"C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv"
OUTPUT_PATH = CSV_PATH  # overwrite in place; change if you prefer a new file
row_idx = 0  # <-- change this if you want a different row

# ====== 1) Load ======
df = pd.read_csv(CSV_PATH)

# ====== 2) Map fault_rating (1..5) -> fault_rating_text ======
fault_map = {
    1: "100% Third-Party at Fault",
    2: "Third-Party Likely at Fault",
    3: "Shared Liability (50/50)",
    4: "Insured at Fault or No third-party",
    5: "Under Investigation or blank"
}

# Ensure integers; coerce invalid to NaN then fill (default to 5 = "Under Investigation or blank")
df["fault_rating"] = pd.to_numeric(df.get("fault_rating", np.nan), errors="coerce")
df["fault_rating"] = df["fault_rating"].fillna(5).astype(int).clip(1, 5)
df["fault_rating_text"] = df["fault_rating"].map(fault_map)

# ====== 3) Helper: build prompt per row for snippet ======
SYSTEM_PROMPT = (
    "You are a concise insurance claims assistant. "
    "Write ultra-short, clear adjuster note snippets."
)

def build_user_prompt(adjuster_notes, witness_available, damage_desc, police_report, fault_text):
    # Guard for Nones/NaNs
    adj = (adjuster_notes or "").strip()
    wit = (witness_available or "").strip()
    dmg = (damage_desc or "").strip()
    police = (police_report or "").strip()
    fault = (fault_text or "").strip()

    # You asked: “First 1–2 sentences will be ultra short and concise coming from summarizing
    # adjuster_notes, witness_available, Damage Description. Last sentence is the exact fault text.”
    # We’ll keep the structure strict.
    return f"""
Summarize briefly (1–2 short sentences) using these fields:
- Adjuster Notes: {adj}
- Witness Available: {wit}
- Damage Description: {dmg}
- Police Report: {police}

THEN add a FINAL sentence that is exactly this phrase (verbatim):
{fault}

Formatting rules:
- Keep the total to 2 short sentences.
- Be specific and neutral, no embellishment.
- If fields are missing, omit them—do not invent.
- Return plain text only (no bullets, no labels).
"""

# ====== 4) Generate adjuster_notes_snippet with LLM ======
# Columns we’ll reference (robust to missing)
col_adjuster_notes = "adjuster_notes"
col_witness = "witness_available"
col_police = "Police Report"
col_damage = "Damage Description" if "Damage Description" in df.columns else "damage_description"

if col_adjuster_notes not in df.columns:
    # Create an empty column to prevent KeyErrors; snippets will still be produced from other fields
    df[col_adjuster_notes] = ""

if col_witness not in df.columns:
    df[col_witness] = ""

if col_damage not in df.columns:
    df[col_damage] = ""

snippets = []

# Simple retry wrapper to handle transient rate limits
def call_llm_with_retry(messages, max_retries=3, backoff_sec=3):
    for attempt in range(max_retries):
        try:
            resp = llm.invoke(messages)
            return resp.content.strip()
        except Exception as e:
            if attempt == max_retries - 1:
                # On final failure, return a fallback
                return "Adjuster notes summary unavailable. " + messages[-1].content.splitlines()[-1].strip()
            time.sleep(backoff_sec * (attempt + 1))

for idx, row in df.iterrows():
    adjuster_notes_val = row.get(col_adjuster_notes, "")
    witness_val = row.get(col_witness, "")
    damage_val = row.get(col_damage, "")
    police_val = row.get(col_police, "")
    fault_text = row.get("fault_rating_text", "")

    user_prompt = build_user_prompt(adjuster_notes_val, witness_val, damage_val, police_val, fault_text)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ]
    snippet = call_llm_with_retry(messages)
    # Make *sure* the last sentence is the fault text (in case the model paraphrased it)
    # We’ll hard-append it if missing.
    if isinstance(snippet, str):
        normalized = snippet.strip()
        if fault_text and fault_text not in normalized:
            # ensure a period before appending
            if not normalized.endswith((".", "!", "?")):
                normalized += "."
            normalized += f" {fault_text}"
        snippets.append(normalized)
    else:
        snippets.append(fault_text or "Under Investigation or blank")

df["adjuster_notes_snippet"] = snippets

# ====== 5) Save ======
df.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Updated file with columns: fault_rating_text, adjuster_notes_snippet\n→ {OUTPUT_PATH}")

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\SujaySunilNagvekar\\VM\\GEN AI\\KM\\claims_with_notes.csv'

In [28]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Updated file with columns: fault_rating_text, adjuster_notes_snippet\n→ {OUTPUT_PATH}")

✅ Updated file with columns: fault_rating_text, adjuster_notes_snippet
→ C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv


In [20]:
row

Claim Number                                                               C9CB6205
Policy Number                                                           PC171779360
Date of Loss                                                               7/5/2023
Time of Loss                                                                4:03:07
Loss Location                     37802 Andre Shoals Suite 964\nSouth Amanda, KY...
Loss Location State                                                        Kentucky
Loss Location Zipcode                                                         89536
Date of Reporting Loss                                                     8/5/2023
Loss cause                                                         Animal Collision
Claimant Name                                                         George Norman
Vehicle Make                                                                   Ford
Vehicle Model                                                               

In [15]:
user_prompt

'\nSummarize briefly (1–2 short sentences) using these fields:\n- Adjuster Notes: **Timestamp:** 2023-10-20 14:23 EST\n**Adjuster Name:** James Smith\n\n- Contacted the claimant, George Norman, on multiple occasions. Initial outreach attempts on 2023-08-06 and 2023-08-07 were unsuccessful. Left voicemails requesting a call back. Successfully connected with the claimant on 2023-08-08.\n- FNOL call transcript reviewed. Claimant expressed confusion about coverage and frustration due to medical and repair delays. Clarified coverage for medical expenses (including surgery) and the need for a vehicle repair estimate and photos for damage assessment.\n- Claimant confirmed the incident involved a deer collision in Kentucky while traveling out of state. Medical documentation, including surgery records for a fractured arm, has been submitted.\n- Advised claimant on next steps for repair process: submitting photos and obtaining repair estimates. Explained deductible responsibilities for vehicle r

In [4]:
import sqlite3

conn = sqlite3.connect("C://Users//SujaySunilNagvekar//VM\GEN AI//KM//vm-GenAI_BI//my_database.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("📋 Tables in database:")
for table in tables:
    print("-", table[0])


📋 Tables in database:
- fnol_data
- policy_data


In [5]:
table_name = "fnol_data"  # or "policy_data"
cursor.execute(f"PRAGMA table_info({table_name});")
columns = cursor.fetchall()

print(f"\n🧾 Columns in '{table_name}':")
for col in columns:
    print("-", col[1])



🧾 Columns in 'fnol_data':
- claim_number
- policy_number
- date_of_loss
- time_of_loss
- loss_location
- loss_location_state
- loss_location_zipcode
- date_of_reporting_loss
- loss_cause
- claimant_name
- vehicle_make
- vehicle_model
- vehicle_year
- damage_description
- reported_by
- claim_status
- claim_assigned
- claim_handler_name
- litigation
- police_report
- photos_videos
- repair_estimate_presence
- repair_estimate
- repair_bill
- towing_receipt
- rental_receipt
- medical_and_injury_documentation
- medical_reports
- hospital_records
- medical_bill
- total_claim_bill
- third_party_information
- subro_opportunity
- third_party_insurance
- third_party_claim_form
- damage_severity
- injury_severity
- date_of_payment_repair
- date_of_payment_medical
- fnol_call
- adjuster_notes


In [6]:
cursor.execute("SELECT subro_opportunity FROM fnol_data limit 5")
rows = cursor.fetchall()

print("\n🔍 Sample repair_bill values:")
for row in rows:
    print(row)



🔍 Sample repair_bill values:
('Yes',)
('Yes',)
('Yes',)
('Yes',)
('Yes',)


In [39]:
cursor.execute("PRAGMA table_info(fnol_data);")
columns = cursor.fetchall()
for col in columns:
    print(col[1])  # Column name


claim_number
policy_number
date_of_loss
time_of_loss
loss_location
loss_location_state
loss_location_zipcode
date_of_reporting_loss
loss_cause
claimant_name
vehicle_make
vehicle_model
vehicle_year
damage_description
reported_by
claim_status
claim_assigned
claim_handler_name
Litigation 
police_report
photos_videos
repair_estimate_presence
repair_estimate
repair_bill
towing_receipt
rental_receipt
medical_and_injury_documentation
medical_reports
hospital_records
Medical bill
total_claim_bill
third_party_information
Subro Opportunity 
third_party_insurance
third_party_claim_form
damage_severity
injury_severity
date_of_payment_repair
date_of_payment_medical
fnol_call
adjuster_notes


In [1]:
import pandas as pd
import numpy as np

path = r"C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv"
df = pd.read_csv(path)

# Optional: make results repeatable
# np.random.seed(42)

# Closed mask (case/whitespace safe)
closed_mask = df["Claim Status"].astype(str).str.strip().str.casefold().eq("closed")

# Assign random scores in 0.1 steps between 0.6 and 0.9
choices = np.array([0.6, 0.7, 0.8, 0.9])
df.loc[closed_mask, "subrogation_score"] = np.random.choice(choices, size=closed_mask.sum())

# Keep one decimal just in case
df["subrogation_score"] = df["subrogation_score"].round(1)

updated = int(closed_mask.sum())
print(f"Updated subrogation_score for {updated} closed claims.")

df.to_csv(path, index=False)
print(f"Saved: {path}")


Updated subrogation_score for 224 closed claims.
Saved: C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv


In [6]:
import pandas as pd
import numpy as np

path = r"C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv"
df = pd.read_csv(path)

# --- Normalize numeric columns you rely on ---
for c in ["Total Claim Bill", "Repair Bill", "Medical bill", "recovery_rate", "recovery_amount"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# If Total Claim Bill missing/NaN, rebuild from parts (Repair + Medical)
rep = df["Repair Bill"].fillna(0) if "Repair Bill" in df.columns else 0
med = df["Medical bill"].fillna(0) if "Medical bill" in df.columns else 0
if "Total Claim Bill" not in df.columns:
    df["Total Claim Bill"] = rep + med
else:
    df["Total Claim Bill"] = df["Total Claim Bill"].fillna(rep + med)

# --- Masks ---
closed_mask = df.get("Claim Status", "").astype(str).str.strip().str.casefold().eq("closed")
# Support either "Subro Opportunity" or "subro_opportunity"
subro_col = "Subro Opportunity" if "Subro Opportunity" in df.columns else ("subro_opportunity" if "subro_opportunity" in df.columns else None)
if subro_col is None:
    raise KeyError("Neither 'Subro Opportunity' nor 'subro_opportunity' column found.")

subro_val = df[subro_col].astype(str).str.strip()
subro_yes_mask = subro_val.str.casefold().eq("yes")
subro_blank_mask = df[subro_col].isna() | (subro_val.eq(""))

# --- 1) For ALL Subro Opportunity = Yes → assign 20–30% and recompute amount ---
df.loc[subro_yes_mask, "recovery_rate"] = np.random.randint(20, 31, size=subro_yes_mask.sum())
df.loc[subro_yes_mask, "recovery_amount"] = (
    df.loc[subro_yes_mask, "Total Claim Bill"].fillna(0) * (df.loc[subro_yes_mask, "recovery_rate"] / 100.0)
)

# --- 2) If Subro Opportunity is blank/Null AND NOT Closed → set both to 0 ---
zero_mask = subro_blank_mask & (~closed_mask)
df.loc[zero_mask, "recovery_rate"] = 0
df.loc[zero_mask, "recovery_amount"] = 0.0

# --- Tidy types ---
df["recovery_rate"] = pd.to_numeric(df["recovery_rate"], errors="coerce").fillna(0).round(0)
df["recovery_amount"] = pd.to_numeric(df["recovery_amount"], errors="coerce").fillna(0).round(2)

# --- Save back ---
df.to_csv(path, index=False)
print(f"Updated recovery_rate (20–30%) and recovery_amount for {int(subro_yes_mask.sum())} 'Subro Opportunity=Yes' claims.")
print(f"Zeroed recovery fields for {int(zero_mask.sum())} rows with blank Subro Opportunity and not closed.")
print(f"Saved: {path}")


Updated recovery_rate (20–30%) and recovery_amount for 200 'Subro Opportunity=Yes' claims.
Zeroed recovery fields for 576 rows with blank Subro Opportunity and not closed.
Saved: C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv


In [7]:
import pandas as pd
import numpy as np

path = r"C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv"
df = pd.read_csv(path)

# --- Normalize numeric fields used below ---
for c in ["Total Claim Bill", "Repair Bill", "Medical bill", "recovery_rate", "recovery_amount"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Rebuild Total Claim Bill if missing/NaN from parts
rep = df["Repair Bill"].fillna(0) if "Repair Bill" in df.columns else 0
med = df["Medical bill"].fillna(0) if "Medical bill" in df.columns else 0
if "Total Claim Bill" not in df.columns:
    df["Total Claim Bill"] = rep + med
else:
    df["Total Claim Bill"] = df["Total Claim Bill"].fillna(rep + med)

# --- Column name for Subro Opportunity (support two variants) ---
subro_col = "Subro Opportunity" if "Subro Opportunity" in df.columns else (
    "subro_opportunity" if "subro_opportunity" in df.columns else "Subro Opportunity"
)
if subro_col not in df.columns:
    df[subro_col] = ""  # create if missing

# --- Masks ---
closed_mask = df.get("Claim Status", "").astype(str).str.strip().str.casefold().eq("closed")
subro_str = df[subro_col].astype(str).str.strip()
blank_mask = df[subro_col].isna() | subro_str.eq("")

# 1) Fill blanks: Closed → Yes, Not Closed → No
df.loc[blank_mask & closed_mask, subro_col] = "Yes"
df.loc[blank_mask & (~closed_mask), subro_col] = "No"

# Recompute masks after fill
subro_str = df[subro_col].astype(str).str.strip()
yes_mask = subro_str.str.casefold().eq("yes")
no_mask  = subro_str.str.casefold().eq("no")

# 2) Apply recovery rules
# For YES → set recovery_rate to 20–30 and recompute amount
df.loc[yes_mask, "recovery_rate"] = np.random.randint(20, 31, size=yes_mask.sum())
df.loc[yes_mask, "recovery_amount"] = (
    df.loc[yes_mask, "Total Claim Bill"].fillna(0) * (df.loc[yes_mask, "recovery_rate"] / 100.0)
)

# For NO → zero out both
df.loc[no_mask, "recovery_rate"] = 0
df.loc[no_mask, "recovery_amount"] = 0.0

# Tidy types
df["recovery_rate"] = pd.to_numeric(df["recovery_rate"], errors="coerce").fillna(0).round(0)
df["recovery_amount"] = pd.to_numeric(df["recovery_amount"], errors="coerce").fillna(0).round(2)

df.to_csv(path, index=False)
print(f"Filled blanks in '{subro_col}' by claim status and updated recovery fields. Saved: {path}")


Filled blanks in 'Subro Opportunity' by claim status and updated recovery fields. Saved: C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv


In [5]:
import pandas as pd
import numpy as np

path = r"C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv"
df = pd.read_csv(path)

# Optional: set seed for reproducibility
# np.random.seed(42)

states = [
    "Colorado", "Idaho", "Kansas", "Montana",
    "Nebraska", "North Dakota", "Oklahoma", "South Dakota"
]

# Replace the entire column with random choices from the list
df["Loss Location State"] = np.random.choice(states, size=len(df))

df.to_csv(path, index=False)
print("Replaced 'Loss Location State' with random values from the 8-state list.")


Replaced 'Loss Location State' with random values from the 8-state list.


In [10]:
import pandas as pd
import numpy as np

PATH = r"C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv"
TARGET_RATE_PCT = 25  # change if needed

df = pd.read_csv(PATH)

for c in ["Total Claim Bill", "Repair Bill", "Medical bill", "recovery_amount"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Rebuild Total Claim Bill if needed from parts
rep = df["Repair Bill"].fillna(0) if "Repair Bill" in df.columns else 0
med = df["Medical bill"].fillna(0) if "Medical bill" in df.columns else 0
if "Total Claim Bill" not in df.columns:
    df["Total Claim Bill"] = rep + med
else:
    df["Total Claim Bill"] = df["Total Claim Bill"].fillna(rep + med)

df["recovery_amount"] = pd.to_numeric(df.get("recovery_amount", 0), errors="coerce").fillna(0)

# ✅ Make recovery_rate = ACTUAL effective rate from amounts
den = df["Total Claim Bill"].replace(0, np.nan)
df["recovery_rate"] = ((df["recovery_amount"] / den) * 100).round(0).fillna(0)  # percent points

# ✅ effective_recovery_rate_pct = same as recovery_rate (keep one source of truth)
df["effective_recovery_rate_pct"] = df["recovery_rate"].astype(float)

# ✅ recovery_gap_amount = target$ − actual$, never negative
target_amount = df["Total Claim Bill"] * (TARGET_RATE_PCT / 100.0)
df["recovery_gap_amount"] = (target_amount - df["recovery_amount"]).clip(lower=0).round(2)

df.to_csv(PATH, index=False)
print("Synced recovery_rate to actuals and recalculated gap. Saved:", PATH)


Synced recovery_rate to actuals and recalculated gap. Saved: C:\Users\SujaySunilNagvekar\VM\GEN AI\KM\claims_with_notes.csv
